# Module 1: Setup Environment and Load Data
There are many ways to load data into Neo4j, and the best method depends on the size and format of your data, as well as your specific use case. We will use the Neo4j Python driver to load data which is a common and flexible method for loading data into Neo4j.

## Setting up the environment
First we need to know where to load the data. We will use a .env file to store our Neo4j connection details. Create a .env file in the root of your project by running the cell below:

In [ ]:
%%bash
if [ ! -f ../.env ]; then
  cp ../.env.example ../.env
fi

When creating the Aura Free instance, you should have downloaded a credentials file. Open that file and copy the content into the .env file you just created. The .env file should look something like this:

```
NEO4J_URI=neo4j+s://12345.databases.neo4j.io
NEO4J_USERNAME=yu35782
NEO4J_PASSWORD=drJ4nK9zkBJ47gnloSPMzROH2
NEO4J_DATABASE=6bcfa123
AURA_INSTANCEID=6bcfa123
AURA_INSTANCENAME=Free instance
```

Now we have our connection details set up, we can use the Neo4j Python driver to connect to our database and load data. We will use the `neo4j` package which provides a simple and efficient way to interact with Neo4j from Python. It has been installed in our Codespace environment, so we can import it and use it to connect to our database.

In [ ]:
%run ../load_data.py

Verify that data got loaded. Notice that we're limiting the results to 25 for better readability. You should see a sample of the data that was loaded into your Neo4j database.

In [ ]:
from neo4j import GraphDatabase, RoutingControl, Result
from neo4j_viz.neo4j import from_neo4j

uri = os.getenv("NEO4J_URI")
user = os.getenv("NEO4J_USERNAME")
password = os.getenv("NEO4J_PASSWORD")
database = os.getenv("NEO4J_DATABASE")

with GraphDatabase.driver(uri=uri, auth=(user, password)) as driver:
    driver.verify_connectivity()
    result = driver.execute_query(
        "MATCH (n)-[r]->(m) RETURN n,r,m LIMIT 25",
        database_=database,
        routing_=RoutingControl.READ,
        result_transformer_=Result.graph,
    )

VG = from_neo4j(result)
VG.render()

Let's count the number of nodes and relationships in our database to verify that the data was loaded correctly. 

In [ ]:
from neo4j import GraphDatabase, RoutingControl, Result

uri = os.getenv("NEO4J_URI")
user = os.getenv("NEO4J_USERNAME")
password = os.getenv("NEO4J_PASSWORD")
database = os.getenv("NEO4J_DATABASE")

with GraphDatabase.driver(uri=uri, auth=(user, password)) as driver:
    driver.verify_connectivity()
    result = driver.execute_query(
        """
        MATCH (n)
        WITH count(n) AS nodes
        MATCH ()-[r]->()
        WITH nodes, count(r) AS relationships
        RETURN nodes, relationships
        """,
        database_=database,
        routing_=RoutingControl.READ,
        result_transformer_=lambda r: [record.data() for record in r],
    )

    for record in result:
        print(record)

If the correct number of nodes (357) and relationships (1048) were loaded proceed to the next module which is about querying the data.